# Calculating RMST Pseudo-Observations

**This notebook calculates RMST pseudo-observations at 1 and 2 years for advanced urthoelial cancer patients receiving pembrolizumab or carboplatin-based chemotherapy.**

In [1]:
import sys
sys.path.append('../..')

import numpy as np
import pandas as pd
from lifelines import KaplanMeierFitter
from lifelines.utils import restricted_mean_survival_time
from joblib import Parallel, delayed

from utils.pseudo_obs import pseudo_observations_rmst

In [2]:
df = pd.read_csv('../outputs/pembro_carbo_features_df.csv')

In [3]:
df = df.set_index('PatientID')

In [4]:
df.shape

(3706, 161)

In [5]:
treatment_df = pd.read_csv('../outputs/pembro_carbo_index.csv')

In [6]:
treatment_df.shape

(3712, 4)

In [7]:
df = pd.merge(df, treatment_df, on = 'PatientID', how = 'left')

In [8]:
df.shape

(3706, 165)

In [9]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [10]:
df['treatment_year'] = df['StartDate'].dt.year

In [11]:
df = df.query('treatment_year <= 2022')

In [12]:
df.shape

(3468, 166)

In [13]:
pseudo_rmst = pseudo_observations_rmst(
    df['duration'].values,
    df['event'].values,
    tau = 365
)

df['rmst_pseudo_1y'] = pseudo_rmst

In [14]:
pseudo_rmst = pseudo_observations_rmst(
    df['duration'].values,
    df['event'].values,
    tau = 730
)

df['rmst_pseudo_2y'] = pseudo_rmst

In [15]:
df = df.reset_index()

In [16]:
df = df[['PatientID', 'rmst_pseudo_1y', 'rmst_pseudo_2y']]

In [17]:
df.head(5)

,PatientID,rmst_pseudo_1y,rmst_pseudo_2y
0,F5AAF96C85477,290.385767,447.209460
1,F412959B03189,369.855120,761.240227
2,F6E944C1709E6,306.039632,287.307633
3,F75087BE5F959,369.855120,763.041424
4,FC0B515A8EBD0,177.524597,167.961183


In [18]:
df.to_csv('../outputs/pseudo_obs_rmst.csv', index = False)